# DeepLabv3+ para segmentacion densa de cultivos

Este notebook documenta el entrenamiento de **un solo modelo**, DeepLabv3+, que
asigna una clase de cultivo a **cada pixel** de una imagen satelital en lugar de
a la parcela completa.

**DeepLabv3+** es una red convolucional con arquitectura encoder-decoder. Su
pieza central es el modulo **ASPP** (Atrous Spatial Pyramid Pooling): un banco
de convoluciones dilatadas a varias escalas que captura contexto fino y grueso
sin perder resolucion espacial. El encoder es **MobileNetV3-Large**, una columna
vertebral ligera pensada para correr rapido incluso sin GPU dedicada.

Trabajamos sobre **PASTIS-R**, un benchmark de parches Sentinel-2 del sur de
Francia donde cada pixel viene anotado con su tipo de cultivo a lo largo de la
temporada. Como DeepLabv3+ es una red 2D y no consume la serie temporal
completa, **colapsamos la dimension temporal por mediana**: las `T` fechas de
cada parche se reducen a una sola imagen de 10 bandas que entra a la red.

El notebook **no entrena en linea**. Dispara la interfaz de linea de comandos de
entrenamiento por subprocess para que la corrida quede registrada en MLflow con
sus metricas y versiones de codigo y datos, y luego lee y muestra ese resultado.

## Requisitos para ejecucion end-to-end

- `data/PASTIS-R/` descomprimido (parches Sentinel-2 + mascaras por pixel).
- Dependencias instaladas via `poetry install --with ml`.
- GPU recomendada para el entrenamiento real; en CPU el modo rapido sigue
  siendo ejecutable.

Si el dataset o las dependencias de entrenamiento no estan disponibles, el
notebook continua en modo degradado: muestra el comando que se habria ejecutado
y un mensaje claro, sin romper la ejecucion.

In [ ]:
# Celda de parametros (papermill). El default es BAJO a proposito para que la
# ejecucion end-to-end sea rapida; el entrenamiento real se lanza con, por
# ejemplo, `-p run_full True -p epochs 30`.
epochs = 2
batch_size = 4
target = "semantic18"
device = "auto"
run_full = False
run_name = "alt-deeplabv3plus-mobilenet-v1"
figures_dir = "paper/figures/us-025"

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
from IPython.display import Markdown, display

# Bootstrap repo root: localiza el pyproject.toml subiendo desde el CWD para
# que el notebook funcione desde notebooks/models/ sin asumir el nombre de
# carpeta.
_REPO_BOOTSTRAP = Path.cwd().resolve()
for _candidate in (_REPO_BOOTSTRAP, *_REPO_BOOTSTRAP.parents):
    if (_candidate / "pyproject.toml").is_file():
        _REPO_BOOTSTRAP = _candidate
        break
if str(_REPO_BOOTSTRAP) not in sys.path:
    sys.path.insert(0, str(_REPO_BOOTSTRAP))

from ml.utils.notebook_setup import find_repo_root
from ml.utils.mlflow_utils import resolve_tracking_uri

# Polars: rendering rico HTML en Jupyter.
pl.Config.set_tbl_formatting("ASCII_MARKDOWN")
pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(60)

%matplotlib inline
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 200

%load_ext autoreload
%autoreload 2

REPO = find_repo_root()
FIGURES = REPO / figures_dir
FIGURES.mkdir(parents=True, exist_ok=True)

# El modelo unico de este notebook. MODEL_KIND es el flag del CLI; RUN_NAME debe
# coincidir con el run que el CLI registra en MLflow para poder recuperarlo.
MODEL_KIND = "deeplabv3plus"
RUN_NAME = run_name
EXPERIMENT_NAME = "agrosat-segmentation"

display(Markdown(
    f"**Configuracion** — modelo: `{MODEL_KIND}` · run: `{RUN_NAME}` · "
    f"epochs: `{epochs}` · target: `{target}` · "
    f"modo: `{'completo' if run_full else 'rapido'}`"
))

## Seccion 1 — La arquitectura

**PASTIS-R** es un benchmark de segmentacion de cultivos: cada muestra es un
parche Sentinel-2 de 128x128 pixeles observado varias veces a lo largo del ano,
con una mascara que etiqueta cada pixel con su clase de cultivo. El reto es
doble: aprovechar la informacion espacial (vecindades, bordes de parcela) y
separar clases que en una sola fecha se ven casi identicas.

**DeepLabv3+** ataca el lado espacial del problema. Su modulo **ASPP** (Atrous
Spatial Pyramid Pooling) aplica varias convoluciones *dilatadas* en paralelo:
cada una mira la imagen con un "campo de vision" distinto (de fino a grueso) sin
reducir la resolucion. Asi la red combina, para cada pixel, su textura local con
el contexto del campo que lo rodea. El decoder despues recupera el detalle de
los bordes para entregar una mascara nitida pixel a pixel.

El encoder es **MobileNetV3-Large**: una columna vertebral ligera (bloques con
convoluciones separables en profundidad y atencion squeeze-and-excitation) que
extrae las features con un costo de computo bajo. Esa eficiencia es lo que
permite entrenar y validar el flujo completo incluso en una laptop sin GPU
dedicada.

**Por que colapsamos el tiempo.** DeepLabv3+ es una red convolucional 2D: espera
una imagen, no una secuencia. PASTIS-R provee `T` fechas por parche, asi que
reducimos esa serie a una sola imagen tomando la **mediana por pixel** sobre el
tiempo (la mediana es robusta a nubes y a fechas atipicas). El resultado es una
imagen de 10 bandas que entra a la red. Esto significa que DeepLabv3+ no razona
sobre el calendario de crecimiento; es deliberadamente nuestro punto de
comparacion "sin tiempo" frente a modelos temporales como TSViT.

El entrenamiento se documenta en MLflow con las metricas mIoU, F1-macro y
exactitud por pixel del mejor epoch de validacion.

In [ ]:
def run_training(n_epochs: int) -> dict[str, float | str | None]:
    """Lanza el CLI de entrenamiento de DeepLabv3+ y parsea sus metricas.

    Invoca `python -m ml.train.train_segmentation` por subprocess para que la
    corrida quede registrada en MLflow. El notebook documenta la invocacion
    CLI: por eso subprocess es la forma correcta aqui, no importar y llamar la
    funcion en linea.

    Args:
        n_epochs: Numero de epochs a entrenar.

    Returns:
        Diccionario con `model`, `miou`, `f1_macro`, `pixel_acc`, `returncode`
        y `error`. Las metricas son `None` si la corrida fallo o no se pudo
        parsear (modo degradado).
    """
    cmd = [
        sys.executable, "-m", "ml.train.train_segmentation",
        "--model", MODEL_KIND,
        "--epochs", str(n_epochs),
        "--batch-size", str(batch_size),
        "--target", target,
        "--device", device,
        "--run-name", RUN_NAME,
    ]
    display(Markdown(f"`{' '.join(cmd)}`"))

    result: dict[str, float | str | None] = {
        "model": MODEL_KIND,
        "miou": None,
        "f1_macro": None,
        "pixel_acc": None,
        "returncode": None,
        "error": None,
    }
    try:
        proc = subprocess.run(
            cmd, cwd=str(REPO), capture_output=True, text=True, check=False
        )
    except OSError as exc:
        result["error"] = f"No se pudo lanzar el subprocess: {exc}"
        print(f"  {MODEL_KIND}: subprocess no disponible ({exc})")
        return result

    result["returncode"] = proc.returncode
    log = (proc.stdout or "") + "\n" + (proc.stderr or "")

    if proc.returncode != 0:
        # Modo degradado: se muestra el final del log pero no se rompe el
        # notebook (el dataset o la GPU pueden no estar disponibles en CI).
        tail = "\n".join(log.strip().splitlines()[-12:])
        result["error"] = f"returncode={proc.returncode}"
        print(f"  {MODEL_KIND}: entrenamiento fallido (returncode={proc.returncode})")
        if tail:
            display(Markdown(f"```\n{tail}\n```"))
        return result

    # El CLI loguea `cli_done` (structlog) con las metricas del mejor epoch.
    # structlog renderiza key=value; parseamos miou/f1_macro/pixel_acc.
    for line in reversed(log.splitlines()):
        if "cli_done" in line:
            for key in ("miou", "f1_macro", "pixel_acc"):
                token = f"{key}="
                if token in line:
                    raw = line.split(token, 1)[1].split()[0].rstrip(",")
                    try:
                        result[key] = float(raw)
                    except ValueError:
                        result[key] = None
            break

    miou = result["miou"]
    if isinstance(miou, float):
        print(f"  {MODEL_KIND}: mIoU={miou:.4f}")
    else:
        print(f"  {MODEL_KIND}: corrida OK pero no se parsearon metricas del log")
    return result

In [ ]:
# En modo rapido se usa el `epochs` bajo de la celda de parametros; en modo
# completo (run_full) se eleva el numero de epochs para una corrida real.
n_epochs = max(epochs, 30) if run_full else epochs

display(Markdown(f"### Entrenando `{MODEL_KIND}` ({n_epochs} epochs)"))
result = run_training(n_epochs)

result_df = pl.DataFrame(
    [result],
    schema={
        "model": pl.Utf8,
        "miou": pl.Float64,
        "f1_macro": pl.Float64,
        "pixel_acc": pl.Float64,
        "returncode": pl.Int64,
        "error": pl.Utf8,
    },
)
display(result_df.select("model", "miou", "f1_macro", "pixel_acc", "returncode"))

## Seccion 2 — Resultados

La corrida quedo registrada en MLflow con sus metricas por epoch y los tags
`code_version` (SHA git) y `data_version` (hash DVC del dataset). Recuperamos el
run del experimento para verificar que el registro funciono y para leer las
metricas del mejor epoch de validacion.

In [ ]:
# Lectura del run registrado. resolve_tracking_uri() elige el servidor MLflow
# local si responde, o el file store ./mlruns como fallback.
mlflow_run_df: pl.DataFrame | None = None
try:
    import mlflow

    mlflow.set_tracking_uri(resolve_tracking_uri())
    exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    if exp is None:
        display(Markdown(
            f"> El experimento `{EXPERIMENT_NAME}` aun no existe en MLflow "
            "(ninguna corrida se registro). Modo degradado."
        ))
    else:
        runs_pd = mlflow.search_runs(
            experiment_ids=[exp.experiment_id],
            filter_string=f"tags.mlflow.runName = '{RUN_NAME}'",
            order_by=["attributes.start_time DESC"],
            max_results=5,
        )
        if runs_pd.empty:
            display(Markdown(
                f"> No hay corridas con run_name `{RUN_NAME}` todavia. Modo degradado."
            ))
        else:
            keep = [
                c for c in (
                    "tags.mlflow.runName",
                    "metrics.best_val_miou",
                    "metrics.best_val_f1_macro",
                    "metrics.best_val_pixel_acc",
                    "tags.code_version",
                    "tags.data_version",
                )
                if c in runs_pd.columns
            ]
            rename = {
                "tags.mlflow.runName": "run_name",
                "metrics.best_val_miou": "miou",
                "metrics.best_val_f1_macro": "f1_macro",
                "metrics.best_val_pixel_acc": "pixel_acc",
                "tags.code_version": "code_version",
                "tags.data_version": "data_version",
            }
            mlflow_run_df = pl.from_pandas(runs_pd[keep]).rename(
                {k: v for k, v in rename.items() if k in keep}
            )
            display(mlflow_run_df)
            if "code_version" in mlflow_run_df.columns and mlflow_run_df.height:
                code_version = mlflow_run_df["code_version"][0]
                display(Markdown(
                    f"Run trazable al commit `code_version={code_version}`."
                ))
except Exception as exc:  # noqa: BLE001 - modo degradado en notebook
    display(Markdown(f"> MLflow no disponible para lectura: `{exc}`. Modo degradado."))

## Conclusiones

Entrenamos DeepLabv3+ para segmentar cultivos **pixel por pixel** sobre parches
satelitales del sur de Francia y registramos la corrida con sus metricas. La
evaluamos con tres varas: **mIoU** (cuanto se solapan la prediccion y la verdad
para cada clase, promediado), **F1-macro** (equilibrio entre aciertos y falsos
positivos, tratando todas las clases por igual) y **exactitud por pixel**
(fraccion de pixeles bien clasificados).

Los numeros concretos salen de la tabla de resultados de este notebook
(`result_df`) y del run recuperado de MLflow. El mIoU y el F1-macro alli
reportados nos dicen, en lenguaje llano, que tan bien la red colorea cada pixel
con su tipo de cultivo cuando solo dispone de la imagen mediana del ano, sin ver
el calendario de crecimiento.

Conviene recordar que estos numeros salen del modo rapido (pocos epochs), pensado
para verificar que el flujo completo corre de principio a fin. No son todavia las
metricas finales: con tan pocas pasadas la red apenas empieza a aprender.

### Lo que sigue

- Lanzar el entrenamiento completo en la laptop (o en una GPU L4/H100) con mas
  epochs (`-p run_full True -p epochs 30`) para obtener las metricas definitivas
  de DeepLabv3+.
- Comparar este segmentador denso "sin tiempo" contra **TSViT**, que si procesa
  la serie temporal completa, para medir cuanta senal aporta el calendario de
  crecimiento al separar cultivos que en una sola fecha se ven casi identicos.